In [85]:
import pandas as pd


In [87]:
file_path = "Data Info.xlsx"
summary_stats_df = pd.read_excel(file_path, sheet_name="Summary Stats")


In [89]:
# Display the number of rows in the original summary_stats_df
print("Number of features in the original dataset:", len(summary_stats_df))


Number of features in the original dataset: 405


# Step 1: High Missingness Variable Removal
Objective: Identify and remove variables with excessive missing data that could hinder meaningful analysis.

Criteria:

High Missingness: Variables with more than 25% missing values (PctMissing > 25%) were removed. These variables are considered unreliable for analysis as they lack sufficient data coverage.
Execution:

Evaluated the PctMissing column to identify variables with more than 25% missing values.
Removed these variables and retained the rest for further analysis.
Results:

Number of Variables Dropped: 291
Examples of Dropped Variables:
Solar_heating_fuel_Idx: 100% missing.
Demo_LaborForce_Unemployed_Vol: 100% missing.
Demo_Pop_Middle_Income_($50k-$149k)_Vol: 24.64% missing, retained (below the threshold).

In [92]:
# Step 1: Remove variables with >50% missing values
high_missing_criteria = summary_stats_df['PctMissing'] > 25
high_missing_variables = summary_stats_df[high_missing_criteria]
filtered_df_step1 = summary_stats_df[~high_missing_criteria]

# Create a new dataframe after filtering
step1_df = summary_stats_df[~high_missing_criteria].copy()

# Display dropped variables for Step 1
print("Dropped Variables in Step 1 (High Missingness):")
print(high_missing_variables)

# Rows removed in Step 1
rows_removed_step1 = len(summary_stats_df) - len(step1_df)
print("Features removed in Step 1 (High Missingness):", rows_removed_step1)

Dropped Variables in Step 1 (High Missingness):
                                       Feature  Count  NumMissing  PctMissing  \
9                      Demo_Pop_Under_10yo_Vol  30709       11813       27.78   
10                   Demo_Pop_GenZ_(10-24)_Vol  31634       10888       25.61   
14                   Demo_Pop_Silent_(75+)_Vol  31114       11408       26.83   
17          Demo_Pop_Upper_Income_($150k+)_Vol  28706       13816       32.49   
18              Demo_LaborForce_Unemployed_Vol      0       42522      100.00   
..                                         ...    ...         ...         ...   
400                     Solar_heating_fuel_Idx      0       42522      100.00   
401            Solar_panel_area_per_capita_Idx      0       42522      100.00   
402     Solar_total_panel_area_residential_Idx      0       42522      100.00   
403  Solar_total_panel_area_nonresidential_Idx      0       42522      100.00   
404   Solar_number_of_system_per_household_Idx      0       4

# Step 2: Low Variance Variable Removal
Objective: Identify and remove variables with very low variance, as they provide minimal differentiation between data points and are unlikely to contribute significantly to modeling EV charger placement.

Criteria:

A minimum standard deviation (Std) threshold of 0.05 was selected.
Variables with Std below this threshold were considered low variance and removed.
This lower threshold allows for the retention of variables with slightly more variability, balancing the need for meaningful features and comprehensive analysis.
Execution:

Evaluated the Std column to identify variables with a standard deviation less than 0.05.
Removed these low-variance variables while retaining the remaining ones in the working dataframe.
Results:

Number of Variables Dropped: 15.
Examples of Dropped Variables:
Demo_Politics_Republican_Vol: Standard deviation = 0, constant value.
VIO_EV_Pct: Standard deviation = 0.013, minimal variation.
Demo_Pop_Growth_2010-20_Vol: Standard deviation = 0.056, previously retained but now removed.
Examples of Retained Variables:

Demo_Pop_Middle_Income_($50k-$149k)_Vol: Significant variability (standard deviation = 2528.2).
Demo_Projected_Pop_Growth_2020-30_Vol: Retained as it exceeds the updated threshold with Std = 0.055.


In [95]:
# Step 2: Remove variables with low variance (threshold = 0.05)
variance_threshold = 0.05

# Identify low-variance variables
low_variance_criteria = step1_df['Std'] < variance_threshold
low_variance_variables = step1_df[low_variance_criteria]

# Create a new dataframe after filtering
step2_df = step1_df[~low_variance_criteria].copy()

# Display dropped variables for Step 2
print("Dropped Variables in Step 2 (Low Variance, Threshold = 0.05):")
print(low_variance_variables)


Dropped Variables in Step 2 (Low Variance, Threshold = 0.05):
                                               Feature  Count  NumMissing  \
27                          Demo_Politics_Democrat_Vol  35913        6609   
28                        Demo_Politics_Republican_Vol  35913        6609   
29                             Demo_Politics_Other_Vol  35913        6609   
173                            Demo_Politics_Other_Pct  32749        9773   
184                                         VIO_EV_Pct  39623        2899   
185                                       VIO_PHEV_Pct  39623        2899   
186                                     VIO_Hybrid_Pct  39623        2899   
187                                VIO_EV + Hybrid_Pct  39623        2899   
188                                      VIO_Total_Pct  39623        2899   
189                                      VIO_Tesla_Pct  39623        2899   
191                                   VIO_Cadillac_Pct  39623        2899   
192           

In [97]:
# Rows removed in Step 2
rows_removed_step2 = len(step1_df) - len(step2_df)
print("Features removed in Step 2 (Low Variance):", rows_removed_step2)

Features removed in Step 2 (Low Variance): 15


In [99]:
# Display all columns of remaining features in step2_df after Step 2
print("Remaining Features in Step 2 (All Columns):")
print(step2_df)


Remaining Features in Step 2 (All Columns):
                Feature  Count  NumMissing  PctMissing       Mean        Std  \
0               Zipcode  42522           0        0.00        NaN        NaN   
1                 State  42522           0        0.00        NaN        NaN   
2        Zip Town State  39623        2899        6.82        NaN        NaN   
3            State_Full  39623        2899        6.82        NaN        NaN   
4                County  41672         850        2.00        NaN        NaN   
..                  ...    ...         ...         ...        ...        ...   
338       VIO_Chevy_Idx  39623        2899        6.82  30.980765  47.231974   
339         VIO_CAR_Idx  39623        2899        6.82  34.803611  60.728346   
340   VIO_CROSSOVER_Idx  39623        2899        6.82  36.297349  59.410414   
341       VIO_TRUCK_Idx  39623        2899        6.82  36.873698  55.538568   
342  VIO_Other Segm_Idx  39623        2899        6.82   5.840611  16.304222

In [101]:
# Export the remaining data after Step 2 to a CSV file
output_file_path = "remaining_data_after_step2.csv"
step2_df.to_csv(output_file_path, index=False)

print(f"Remaining data after Step 2 has been exported to: {output_file_path}")

Remaining data after Step 2 has been exported to: remaining_data_after_step2.csv
